<a href="https://colab.research.google.com/github/kasturikirankumar1101-lab/AI_TOOLS/blob/main/CodeConverter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Code Converter — Production-Grade Gradio App
Converts source code between languages using OpenAI GPT API.
Runs in Google Colab (reads key from Colab Secrets) or locally (reads from env var).
"""

import os
import re
import logging
import gradio as gr
from openai import OpenAI, APIError, APIConnectionError, RateLimitError, AuthenticationError

# ── Colab Secrets (preferred) with os.environ fallback ────────────────────────
try:
    from google.colab import userdata as colab_userdata
    _COLAB = True
except ImportError:
    _COLAB = False

# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

if _COLAB:
    logger.info("Running inside Google Colab — API key will be read from Colab Secrets.")
else:
    logger.info("Running outside Google Colab — API key will be read from environment variable.")

# ── Constants ──────────────────────────────────────────────────────────────────
LANGUAGES      = ["Java", "Python", "Javascript", "C++", "COBOL", "VC++", ".NET"]
MODEL          = "gpt-4o"    # change to "gpt-4-turbo" or "gpt-3.5-turbo" if needed
MAX_TOKENS     = 8192        # enough for large file conversions
MAX_FILE_CHARS = 100_000     # ~100 KB safety cap on input


# ── API Client ─────────────────────────────────────────────────────────────────
def get_client() -> OpenAI:
    """
    Resolve the OpenAI API key and return an OpenAI client.

    Resolution order:
      1. Google Colab Secrets  (key name: OPENAI_API_KEY)  — when running in Colab
      2. Environment variable   OPENAI_API_KEY             — fallback / local runs

    To add the key in Colab:
      - Click the 🔑 key icon in the left sidebar
      - Add a secret named  OPENAI_API_KEY  with your key value
      - Toggle "Notebook access" ON
    """
    api_key = ""

    # 1️⃣  Try Colab Secrets first
    if _COLAB:
        try:
            api_key = colab_userdata.get("OPENAI_API_KEY").strip()
            logger.info("✅ API key loaded from Colab Secrets.")
        except Exception as e:
            logger.warning("Colab Secrets lookup failed (%s). Falling back to env var.", e)

    # 2️⃣  Fall back to environment variable
    if not api_key:
        api_key = os.environ.get("OPENAI_API_KEY", "").strip()
        if api_key:
            logger.info("✅ API key loaded from environment variable.")

    # 3️⃣  Neither source had a key — raise a clear error
    if not api_key:
        raise EnvironmentError(
            "OPENAI_API_KEY not found.\n"
            "• In Colab : open the 🔑 Secrets panel and add OPENAI_API_KEY.\n"
            "• Locally  : run  export OPENAI_API_KEY=sk-...  before launching."
        )

    if not api_key.startswith("sk-"):
        logger.warning("OPENAI_API_KEY does not start with 'sk-' — double-check the value.")

    return OpenAI(api_key=api_key)


# ── Prompt Builders ────────────────────────────────────────────────────────────
def build_prompts(source_language: str, target_language: str, code_snippet: str) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) with all placeholders resolved."""

    system_prompt = f"""You are a senior software engineer and expert code translator \
specializing in multi-language systems.

Objective:
Accurately convert code from {source_language} to {target_language} while preserving \
logic, behavior, and intent.

Core Requirements:
- Preserve 100% of the original logic and functionality.
- Do NOT introduce behavioral changes.
- Translate code into idiomatic, production-quality {target_language}.
- Follow best practices, naming conventions, and standard libraries of {target_language}.
- Maintain equivalent performance characteristics where possible.

Strict Rules:
- Do NOT omit any part of the code.
- Do NOT hallucinate libraries or APIs.
- If a feature is unsupported in {target_language}, provide the closest valid alternative.
- Clearly mark assumptions using: // ASSUMPTION: <text>
- Clearly mark limitations using: // LIMITATION: <text>
- Keep output deterministic and structured.

Output Format (MANDATORY):
Inside <converted_code>, output ONLY raw code — no markdown fences, no explanations, \
no extra text before or after.

<converted_code language="{target_language}">
code here
</converted_code>

Edge Case Handling:
- Handle language-specific differences (typing, memory management, async behaviour).
- Convert standard libraries appropriately.
- Preserve error-handling semantics.
- Ensure correct data structures and types."""

    user_prompt = f"""Task:
Convert the following code from {source_language} to {target_language}.

Conversion Requirements:
- Preserve logic exactly.
- Ensure idiomatic {target_language} implementation.
- Maintain readability and maintainability.
- Include inline comments where necessary.

Input Code ({source_language}):
{code_snippet}"""

    return system_prompt, user_prompt


# ── Model Call ─────────────────────────────────────────────────────────────────
def call_model(system_prompt: str, user_prompt: str) -> str:
    """
    Call the OpenAI API and return the assistant's text response.
    Raises descriptive RuntimeError on failure.
    """
    client = get_client()

    try:
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
        )
    except AuthenticationError:
        raise RuntimeError(
            "❌ Invalid OpenAI API key.\n"
            "Please check your key at https://platform.openai.com/api-keys"
        )
    except RateLimitError:
        raise RuntimeError(
            "⏳ Rate limit or quota reached.\n"
            "Please check your usage at https://platform.openai.com/usage"
        )
    except APIConnectionError:
        raise RuntimeError("Could not connect to the OpenAI API. Check your network.")
    except APIError as e:
        if "insufficient_quota" in str(e) or "billing" in str(e).lower():
            raise RuntimeError(
                "💳 Your OpenAI credit balance is too low.\n"
                "Please visit https://platform.openai.com/settings/billing to add credits."
            )
        raise RuntimeError(f"OpenAI API error: {e}")

    # Extract text from the response
    return response.choices[0].message.content or ""


# ── Response Parser ────────────────────────────────────────────────────────────
def extract_converted_code(raw_response: str) -> str:
    """
    Pull the code out of the <converted_code ...>...</converted_code> tag.
    Falls back to returning the full response if the tag is absent.
    """
    match = re.search(
        r"<converted_code[^>]*>(.*?)</converted_code>",
        raw_response,
        re.DOTALL,
    )
    if match:
        return match.group(1).strip()
    logger.warning("Could not find <converted_code> tag in model response; returning raw output.")
    return raw_response.strip()


# ── Dynamic Textbox Helper ─────────────────────────────────────────────────────
def dynamic_update(text: str, min_lines: int = 5, max_lines: int = 60) -> gr.update:
    """Return a gr.update that sets value and auto-sizes the Textbox."""
    lines = text.splitlines()
    display_lines = sum(max(1, (len(l) // 100) + 1) for l in lines)
    clamped = max(min_lines, min(display_lines, max_lines))
    return gr.update(value=text, lines=clamped)


# ── Main Processing Function ───────────────────────────────────────────────────
def process_file(
    file,
    convert_from: str,
    convert_to: str,
    state: dict,          # gr.State — persists file cache across calls in the same session
) -> tuple:
    """
    Gradio handler. Returns (output_update, new_state).

    Caching logic:
      - If the same file is submitted again with only the target language changed,
        the file is NOT re-read from disk; the cached content is reused.
      - Cache is invalidated when a new file is uploaded.
    """

    # ── Input validation ───────────────────────────────────────────────────────
    if file is None:
        return dynamic_update("⚠️  No file selected. Please upload a file and try again."), state
    if not convert_from:
        return dynamic_update("⚠️  Please select a 'Convert From' language."), state
    if not convert_to:
        return dynamic_update("⚠️  Please select a 'Convert To' language."), state
    if convert_from == convert_to:
        return dynamic_update(
            "⚠️  'Convert From' and 'Convert To' are the same language. "
            "Please select different languages."
        ), state

    # ── File read (with caching) ───────────────────────────────────────────────
    cached_path    = state.get("file_path")
    cached_content = state.get("file_content")

    if file.name != cached_path:
        logger.info("New file detected — reading from disk: %s", file.name)
        try:
            with open(file.name, "r", encoding="utf-8", errors="replace") as f:
                content = f.read()
        except OSError as e:
            return dynamic_update(f"❌ Could not read file:\n{e}"), state

        if len(content) > MAX_FILE_CHARS:
            return dynamic_update(
                f"⚠️  File is too large ({len(content):,} chars). "
                f"Maximum allowed is {MAX_FILE_CHARS:,} chars."
            ), state

        state = {"file_path": file.name, "file_content": content}
        logger.info("File cached (%d chars).", len(content))
    else:
        logger.info("Cache hit — reusing content for: %s", file.name)
        content = cached_content

    # ── Build prompts & call model ─────────────────────────────────────────────
    try:
        system_prompt, user_prompt = build_prompts(convert_from, convert_to, content)
        logger.info("Calling model: %s → %s", convert_from, convert_to)
        raw_response   = call_model(system_prompt, user_prompt)
        converted_code = extract_converted_code(raw_response)
        logger.info("Conversion complete (%d chars).", len(converted_code))
    except RuntimeError as e:
        return dynamic_update(f"❌ {e}"), state
    except Exception as e:
        logger.exception("Unexpected error during conversion.")
        return dynamic_update(f"❌ Unexpected error:\n{e}"), state

    return dynamic_update(converted_code), state


# ── UI Layout ──────────────────────────────────────────────────────────────────
with gr.Blocks(title="Code Converter", theme=gr.themes.Soft()) as demo:

    # Session-scoped cache: {"file_path": str, "file_content": str}
    session_state = gr.State({})

    gr.Markdown(
        """
        # 🔄 Code Converter
        Upload a source file, choose the languages, and click **Convert**.
        The output area resizes automatically to fit the result.
        """
    )

    with gr.Row():
        # ── Left panel: inputs ─────────────────────────────────────────────────
        with gr.Column(scale=1):
            file_input = gr.File(
                label="Select File",
                file_count="single",
            )
            convert_from = gr.Dropdown(
                label="Convert From",
                choices=LANGUAGES,
                value=None,
                interactive=True,
            )
            convert_to = gr.Dropdown(
                label="Convert To",
                choices=LANGUAGES,
                value=None,
                interactive=True,
            )
            submit_btn = gr.Button("Convert", variant="primary", size="lg")

        # ── Right panel: output ────────────────────────────────────────────────
        with gr.Column(scale=2):
            output_box = gr.Textbox(
                label="Converted Code",
                lines=5,
                max_lines=60,
                interactive=False,
                placeholder="Converted code will appear here…",
                show_copy_button=True,
            )

    # ── Event wiring ───────────────────────────────────────────────────────────
    submit_btn.click(
        fn=process_file,
        inputs=[file_input, convert_from, convert_to, session_state],
        outputs=[output_box, session_state],
    )

    gr.Markdown(
        """<sub>
        🔑 <b>Colab users</b>: add <code>OPENAI_API_KEY</code> in the Secrets panel (left sidebar) and enable Notebook access.<br>
        💻 <b>Local users</b>: run <code>export OPENAI_API_KEY=sk-...</code> before launching.<br>
        🤖 Model in use: <code>gpt-4o</code> — change the <code>MODEL</code> constant at the top of the file to switch models.
        </sub>"""
    )


# In Colab, share=True creates a public tunnel URL.
# server_port is omitted so Gradio auto-selects a free port.
demo.launch(
    share=True,   # set False if running locally; share=True gives a public tunnel URL in Colab
)

In [1]:
"""
Code Converter — Production-Grade Gradio App
Converts source code between languages using OpenAI GPT API.
Runs in Google Colab (reads key from Colab Secrets) or locally (reads from env var).
"""

import os
import re
import logging
import gradio as gr
from openai import OpenAI, APIError, APIConnectionError, RateLimitError, AuthenticationError

# ── Colab Secrets (preferred) with os.environ fallback ────────────────────────
try:
    from google.colab import userdata as colab_userdata
    _COLAB = True
except ImportError:
    _COLAB = False

# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

if _COLAB:
    logger.info("Running inside Google Colab — API key will be read from Colab Secrets.")
else:
    logger.info("Running outside Google Colab — API key will be read from environment variable.")

# ── Constants ──────────────────────────────────────────────────────────────────
LANGUAGES      = ["Java", "Python", "Javascript", "C++", "COBOL", "VC++", ".NET"]
MODEL          = "gpt-4o"    # change to "gpt-4-turbo" or "gpt-3.5-turbo" if needed
MAX_TOKENS     = 8192        # enough for large file conversions
MAX_INPUT_CHARS = 100_000    # ~100 KB safety cap on input


# ── API Client ─────────────────────────────────────────────────────────────────
def get_client() -> OpenAI:
    """
    Resolve the OpenAI API key and return an OpenAI client.

    Resolution order:
      1. Google Colab Secrets  (key name: OPENAI_API_KEY)  — when running in Colab
      2. Environment variable   OPENAI_API_KEY             — fallback / local runs

    To add the key in Colab:
      - Click the 🔑 key icon in the left sidebar
      - Add a secret named  OPENAI_API_KEY  with your key value
      - Toggle "Notebook access" ON
    """
    api_key = ""

    # 1️⃣  Try Colab Secrets first
    if _COLAB:
        try:
            api_key = colab_userdata.get("OPENAI_API_KEY").strip()
            logger.info("✅ API key loaded from Colab Secrets.")
        except Exception as e:
            logger.warning("Colab Secrets lookup failed (%s). Falling back to env var.", e)

    # 2️⃣  Fall back to environment variable
    if not api_key:
        api_key = os.environ.get("OPENAI_API_KEY", "").strip()
        if api_key:
            logger.info("✅ API key loaded from environment variable.")

    # 3️⃣  Neither source had a key — raise a clear error
    if not api_key:
        raise EnvironmentError(
            "OPENAI_API_KEY not found.\n"
            "• In Colab : open the 🔑 Secrets panel and add OPENAI_API_KEY.\n"
            "• Locally  : run  export OPENAI_API_KEY=sk-...  before launching."
        )

    if not api_key.startswith("sk-"):
        logger.warning("OPENAI_API_KEY does not start with 'sk-' — double-check the value.")

    return OpenAI(api_key=api_key)


# ── Prompt Builders ────────────────────────────────────────────────────────────
def build_prompts(source_language: str, target_language: str, code_snippet: str) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) with all placeholders resolved."""

    system_prompt = f"""You are a senior software engineer and expert code translator \
specializing in multi-language systems.

Objective:
Accurately convert code from {source_language} to {target_language} while preserving \
logic, behavior, and intent.

Core Requirements:
- Preserve 100% of the original logic and functionality.
- Do NOT introduce behavioral changes.
- Translate code into idiomatic, production-quality {target_language}.
- Follow best practices, naming conventions, and standard libraries of {target_language}.
- Maintain equivalent performance characteristics where possible.

Strict Rules:
- Do NOT omit any part of the code.
- Do NOT hallucinate libraries or APIs.
- If a feature is unsupported in {target_language}, provide the closest valid alternative.
- Clearly mark assumptions using: // ASSUMPTION: <text>
- Clearly mark limitations using: // LIMITATION: <text>
- Keep output deterministic and structured.

Output Format (MANDATORY):
Inside <converted_code>, output ONLY raw code — no markdown fences, no explanations, \
no extra text before or after.

<converted_code language="{target_language}">
code here
</converted_code>

Edge Case Handling:
- Handle language-specific differences (typing, memory management, async behaviour).
- Convert standard libraries appropriately.
- Preserve error-handling semantics.
- Ensure correct data structures and types."""

    user_prompt = f"""Task:
Convert the following code from {source_language} to {target_language}.

Conversion Requirements:
- Preserve logic exactly.
- Ensure idiomatic {target_language} implementation.
- Maintain readability and maintainability.
- Include inline comments where necessary.

Input Code ({source_language}):
{code_snippet}"""

    return system_prompt, user_prompt


# ── Model Call ─────────────────────────────────────────────────────────────────
def call_model(system_prompt: str, user_prompt: str) -> str:
    """
    Call the OpenAI API and return the assistant's text response.
    Raises descriptive RuntimeError on failure.
    """
    client = get_client()

    try:
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
        )
    except AuthenticationError:
        raise RuntimeError(
            "❌ Invalid OpenAI API key.\n"
            "Please check your key at https://platform.openai.com/api-keys"
        )
    except RateLimitError:
        raise RuntimeError(
            "⏳ Rate limit or quota reached.\n"
            "Please check your usage at https://platform.openai.com/usage"
        )
    except APIConnectionError:
        raise RuntimeError("Could not connect to the OpenAI API. Check your network.")
    except APIError as e:
        if "insufficient_quota" in str(e) or "billing" in str(e).lower():
            raise RuntimeError(
                "💳 Your OpenAI credit balance is too low.\n"
                "Please visit https://platform.openai.com/settings/billing to add credits."
            )
        raise RuntimeError(f"OpenAI API error: {e}")

    # Extract text from the response
    return response.choices[0].message.content or ""


# ── Response Parser ────────────────────────────────────────────────────────────
def extract_converted_code(raw_response: str) -> str:
    """
    Pull the code out of the <converted_code ...>...</converted_code> tag.
    Falls back to returning the full response if the tag is absent.
    """
    match = re.search(
        r"<converted_code[^>]*>(.*?)</converted_code>",
        raw_response,
        re.DOTALL,
    )
    if match:
        return match.group(1).strip()
    logger.warning("Could not find <converted_code> tag in model response; returning raw output.")
    return raw_response.strip()


# ── Dynamic Textbox Helper ─────────────────────────────────────────────────────
def dynamic_update(text: str, min_lines: int = 5, max_lines: int = 60) -> gr.update:
    """Return a gr.update that sets value and auto-sizes the Textbox."""
    lines = text.splitlines()
    display_lines = sum(max(1, (len(l) // 100) + 1) for l in lines)
    clamped = max(min_lines, min(display_lines, max_lines))
    return gr.update(value=text, lines=clamped)


# ── Main Processing Function ───────────────────────────────────────────────────
def process_code(
    code_input: str,
    convert_from: str,
    convert_to: str,
) -> gr.update:
    """
    Gradio handler. Accepts raw code from the input Textbox,
    converts it using the GPT model, and returns a dynamic Textbox update.
    """

    # ── Input validation ───────────────────────────────────────────────────────
    if not code_input or not code_input.strip():
        return dynamic_update("⚠️  Input is empty. Please paste your source code and try again.")
    if not convert_from:
        return dynamic_update("⚠️  Please select a 'Convert From' language.")
    if not convert_to:
        return dynamic_update("⚠️  Please select a 'Convert To' language.")
    if convert_from == convert_to:
        return dynamic_update(
            "⚠️  'Convert From' and 'Convert To' are the same language. "
            "Please select different languages."
        )
    if len(code_input) > MAX_INPUT_CHARS:
        return dynamic_update(
            f"⚠️  Input is too large ({len(code_input):,} chars). "
            f"Maximum allowed is {MAX_INPUT_CHARS:,} chars."
        )

    # ── Build prompts & call model ─────────────────────────────────────────────
    try:
        system_prompt, user_prompt = build_prompts(convert_from, convert_to, code_input.strip())
        logger.info("Calling model: %s → %s (%d chars)", convert_from, convert_to, len(code_input))
        raw_response   = call_model(system_prompt, user_prompt)
        converted_code = extract_converted_code(raw_response)
        logger.info("Conversion complete (%d chars).", len(converted_code))
    except RuntimeError as e:
        return dynamic_update(f"❌ {e}")
    except Exception as e:
        logger.exception("Unexpected error during conversion.")
        return dynamic_update(f"❌ Unexpected error:\n{e}")

    return dynamic_update(converted_code)


# ── UI Layout ──────────────────────────────────────────────────────────────────
with gr.Blocks(title="Code Converter", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # 🔄 Code Converter
        Paste your source code, choose the languages, and click **Convert**.
        The output area resizes automatically to fit the result.
        """
    )

    with gr.Row():
        # ── Left panel: dropdowns + input code ────────────────────────────────
        with gr.Column(scale=1):
            convert_from = gr.Dropdown(
                label="Convert From",
                choices=LANGUAGES,
                value=None,
                interactive=True,
            )
            convert_to = gr.Dropdown(
                label="Convert To",
                choices=LANGUAGES,
                value=None,
                interactive=True,
            )
            code_input = gr.Textbox(
                label="Source Code",
                lines=20,
                max_lines=60,
                interactive=True,
                placeholder="Paste your source code here…",
                show_copy_button=True,
            )
            submit_btn = gr.Button("Convert", variant="primary", size="lg")

        # ── Right panel: converted output ──────────────────────────────────────
        with gr.Column(scale=1):
            output_box = gr.Textbox(
                label="Converted Code",
                lines=20,
                max_lines=60,
                interactive=False,
                placeholder="Converted code will appear here…",
                show_copy_button=True,
            )

    # ── Event wiring ───────────────────────────────────────────────────────────
    submit_btn.click(
        fn=process_code,
        inputs=[code_input, convert_from, convert_to],
        outputs=output_box,
    )

    gr.Markdown(
        """<sub>
        🔑 <b>Colab users</b>: add <code>OPENAI_API_KEY</code> in the Secrets panel (left sidebar) and enable Notebook access.<br>
        💻 <b>Local users</b>: run <code>export OPENAI_API_KEY=sk-...</code> before launching.<br>
        🤖 Model in use: <code>gpt-4o</code> — change the <code>MODEL</code> constant at the top of the file to switch models.
        </sub>"""
    )


# In Colab, share=True creates a public tunnel URL.
# server_port is omitted so Gradio auto-selects a free port.
demo.launch(
    share=True,   # set False if running locally; share=True gives a public tunnel URL in Colab
)

/tmp/ipykernel_32357/1644309207.py:251: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Code Converter", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://02a2f1c23813464673.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [2]:
users = [
    {"name": "Alice", "age": 25},
    {"name": "Bob", "age": 17},
    {"name": "Charlie", "age": 30}
]

def get_adults(users):
    # Filter users who are adults (age >= 18)
    # and collect their names
    return [user["name"] for user in users if user["age"] >= 18]

print(get_adults(users))

['Alice', 'Charlie']
